# 1.3 矩阵乘法与 set_cube_tile_shapes

上一节学习的逐元素算子主要依赖向量计算，每个输出位置通常只依赖对应位置的输入。本节进入另一类非常重要的计算：矩阵乘法。矩阵乘法是深度学习模型中的核心计算，线性层、MLP、Attention Score 都会大量使用它。

矩阵乘法和逐元素计算最大的区别是：一个输出元素不是由一个输入位置直接算出来，而是由左矩阵的一整行和右矩阵的一整列做乘加得到。因此，本节的重点不只是写出 `pypto.matmul`，还要看懂 `[M, K] @ [K, N] -> [M, N]` 这条 shape 规则，以及为什么它使用 `set_cube_tile_shapes`。

本节覆盖矩阵乘法的完整 beginner 用例：基础二维矩阵乘法、Batch MatMul、Batch Broadcast MatMul、左/右输入转置、MatMul + Bias，以及 MatMul + Bias + ReLU 融合扩展。每个代码模块都包含独立的 main 调用和验证输出。

---
## 前置说明

本节在第一个代码单元格中集中放置环境准备代码（import、设备选择、`RUN_MODE` 等）。后续每个可运行模块在定义 kernel 前只调用 `reset_pypto_notebook_state()` 重置编译状态，不再重复完整的环境初始化。

环境准备的具体含义参见 `01.01_chapter_intro.ipynb` §1。


## 直观理解：矩阵乘法不是逐元素乘法

逐元素乘法是同一位置上的两个元素相乘；矩阵乘法不同，它会将左矩阵的一行与右矩阵的一列配对，对对应元素做乘法后再累加。

例如：

```python
a 的一行 = [a1, a2, a3]
b 的一列 = [b1, b2, b3]
```

矩阵乘法中一个输出元素是：

```python
a1 * b1 + a2 * b2 + a3 * b3
```

所以矩阵乘法比逐元素计算复杂得多，也更依赖 NPU 的矩阵计算能力。这就是为什么本节使用 `set_cube_tile_shapes`，而不是上一节的 `set_vec_tile_shapes`。


## 2. 矩阵乘法的形状规则

对于二维矩阵乘法：

```python
out = a @ b
```

如果 `a` 的形状是 `[M, K]`，`b` 的形状是 `[K, N]`，那么输出 `out` 的形状是 `[M, N]`。

其中：

- `M` 表示左矩阵的行数。
- `K` 是乘法累加维度。
- `N` 表示右矩阵的列数。

理解这三个维度非常重要，因为后续的 Cube Tile 设置、矩阵切分和 Attention Score 都离不开 `M、K、N`。


可以把 `[M, K] @ [K, N] -> [M, N]` 记成一句话：

“左边有 M 行，右边有 N 列，所以结果是 M 行 N 列；中间那个 K 是用来配对相乘再累加的，最后会被消掉。”

例如 `[64, 128] @ [128, 64]`：

- 左边有 64 行。
- 右边有 64 列。
- 中间的 128 对齐，负责乘加累积。
- 所以结果是 `[64, 64]`。


## 3. 为什么使用 set_cube_tile_shapes

逐元素计算通常适合向量类执行，而矩阵乘法会涉及大量乘加累积，更适合使用 NPU 的矩阵计算能力。PyPTO 中一般使用 `pypto.set_cube_tile_shapes` 为矩阵乘法类计算设置 Cube Tile。

示例：

```python
pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
```

初学阶段可以先这样理解：

- 它不改变 `a @ b` 的数学结果。
- 它告诉编译系统矩阵乘法可以如何分块。
- 它和向量 Tile 面向的计算类型不同。

也就是说，`set_cube_tile_shapes` 解决的是矩阵计算的执行组织问题。


## 3.1 矩阵乘法的常见模式

矩阵乘法不只是一条 `[M, K] @ [K, N] -> [M, N]` 规则。实际使用时，还会遇到 batch 维、batch 广播、输入转置和 bias 融合。它们都可以按同一个思路理解：最后两维负责矩阵乘法，前面的维度负责 batch 或广播。

| 模式 | Shape 或表达式 | 含义 |
| --- | --- | --- |
| 普通 MatMul | `[M, K] @ [K, N] -> [M, N]` | 单个二维矩阵乘法 |
| Batch MatMul | `[B, M, K] @ [B, K, N] -> [B, M, N]` | 每个 batch 独立做矩阵乘法 |
| Broadcast MatMul | `[1, M, K] @ [B, K, N] -> [B, M, N]` | batch 维度按广播规则扩展 |
| 右矩阵转置 | `a @ b.T` | `b_trans=True` 时先按逻辑转置右输入 |
| 左矩阵转置 | `a.T @ b` | `a_trans=True` 时先按逻辑转置左输入 |
| MatMul + Bias | `[M, K] @ [K, N] + [1, N] -> [M, N]` | bias 沿输出行方向广播 |

判断转置时，先在纸面上写出逻辑 shape。例如 `b_trans=True` 时，把 `b.shape=[N, K]` 先看成 `[K, N]`，再套用普通矩阵乘法规则。

## 4. 编写矩阵乘法算子

下面先实现一个基础矩阵乘法算子。输入 `a` 和 `b` 分别是两个矩阵，输出 `out` 保存矩阵乘法结果。


## 4.1 基础矩阵乘法算子规格

在写矩阵乘法 Kernel 之前，先明确输入输出规格。

| 项目 | 说明 |
| --- | --- |
| 数学表达式 | `out = a @ b` |
| 输入 `a` | FP32 Tensor，shape 为 `[M, K]` |
| 输入 `b` | FP32 Tensor，shape 为 `[K, N]` |
| 输出 `out` | FP32 Tensor，shape 为 `[M, N]` |
| 规约维度 | `K`，在乘加过程中被累加消去 |
| 计算类型 | Cube 类矩阵乘法 |
| Tile 设置 | `pypto.set_cube_tile_shapes(...)` |

本模块使用确定的小矩阵输入，便于直接对照手算结果：`[[1, 2], [3, 4]] @ [[5, 6], [7, 8]] = [[19, 22], [43, 50]]`。

在 Notebook 中，每个可运行模块都先清理 PyPTO 的记录状态，再定义当前模块的 JIT kernel。这样一个模块的失败不会影响下一个模块的验证。

In [ ]:
import os

# 环境准备：参见 01.01_chapter_intro.ipynb §1 的详细说明
# ASCEND_RT_VISIBLE_DEVICES 必须在 import torch 之前设置才生效。
# 如需指定其他 NPU，请在启动 Notebook 前设置 ASCEND_RT_VISIBLE_DEVICES。
os.environ.setdefault("ASCEND_RT_VISIBLE_DEVICES", "13")

import torch
import pypto

try:
    import torch_npu  # noqa: F401
except Exception:
    torch_npu = None


def reset_pypto_notebook_state():
    "Clean stale PyPTO recording state before defining a JIT kernel."
    try:
        pypto.reset()
    except Exception:
        pass

    try:
        from pypto._controller import Controller
        Controller.end_function()
    except Exception:
        pass


def get_device():
    if torch_npu is None:
        return "cpu"

    device_id = int(os.environ.get("TILE_FWK_DEVICE_ID", "0"))
    try:
        torch.npu.set_device(device_id)
    except Exception as exc:
        print("NPU device is not ready:", exc)
        return "cpu"
    return f"npu:{device_id}"


reset_pypto_notebook_state()
device = get_device()
RUN_MODE = pypto.RunMode.NPU if device != "cpu" else pypto.RunMode.SIM

print("ASCEND_RT_VISIBLE_DEVICES:", os.environ.get("ASCEND_RT_VISIBLE_DEVICES", "<未设置>"))
print("TILE_FWK_DEVICE_ID:", os.environ.get("TILE_FWK_DEVICE_ID", "<未设置，默认 0>"))
print("device:", device)

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def matmul_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    out.move(pypto.matmul(a, b, pypto.DT_FP32))


def main_matmul():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    a = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32, device=device)
    b = torch.tensor([[5, 6], [7, 8]], dtype=torch.float32, device=device)
    out = torch.empty((2, 2), dtype=torch.float32, device=device)

    matmul_kernel(a, b, out)
    ref = torch.matmul(a, b)

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-3, atol=1e-3)
    print("matmul_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("a:", a.cpu())
    print("b:", b.cpu())
    print("输出:", out.cpu())
    print("参考:", ref.cpu())
    print("最大误差:", max_diff)


main_matmul()


`matmul_kernel` 是矩阵乘法 kernel。`set_cube_tile_shapes` 只影响矩阵块如何组织执行；真正的数学关系仍然是 `[M, K] @ [K, N] -> [M, N]`，结果通过 `out.move` 或 `out[:]` 写回。


### 代码细节解释

- `set_cube_tile_shapes([32, 32], [64, 64], [64, 64])`：为矩阵乘法设置 Cube Tile。它面向矩阵乘法的分块执行，不等同于输入 Tensor 的完整 shape。
- `pypto.matmul(a, b, out_dtype=out.dtype)`：表达矩阵乘法，并将输出 dtype 与 `out` 对齐。
- `out.move(...)`：把矩阵乘法表达式结果写回输出 Tensor。

由于 BF16 的数值精度低于 FP32，验证时误差阈值通常需要比 FP32 更宽松。


### 再把 matmul_kernel 解释为自然语言

这段 Kernel 做的事是：

“给我矩阵 `a` 和矩阵 `b`，我用矩阵乘法规则计算 `a @ b`，然后把结果放进 `out`。”

其中：

- `a` 和 `b` 是输入 Tensor。
- `out` 是输出 Tensor。
- `pypto.matmul(...)` 是真正表达矩阵乘法的地方。
- `out_dtype=out.dtype` 表示输出数字类型跟 `out` 保持一致。
- `set_cube_tile_shapes(...)` 是告诉 PyPTO 这是一类适合按矩阵块组织的计算。


这里有两个点需要注意：

1. `pypto.matmul(a, b, out_dtype=out.dtype)` 明确指定矩阵乘法输出 dtype。
2. 输出写回使用 `out.move(...)`，表示将矩阵乘法结果写入调用者传入的输出 Tensor。

下面的代码模块会构造 `[64, 128] @ [128, 64] -> [64, 64]` 的输入，调用 kernel，并用 `torch.matmul` 做参考验证。

这个单元用来验证 `matmul_kernel`。它先在当前设备上构造一组小尺寸输入，再提前分配输出 Tensor 并调用 PyPTO kernel。随后用普通 PyTorch 写出等价的 矩阵乘法 参考结果，打印 shape、样例值和最大误差；最后的 `assert_close` 是正式检查，误差超出阈值时会直接报错。


**预期输出说明**

运行成功后，会看到 `matmul_kernel 验证通过`，并打印 `a shape`、`b shape`、输出 shape、最大误差和输出前几项。


## 5. Batch MatMul：前面的维度表示批次

当输入是 `[B, M, K]` 和 `[B, K, N]` 时，最后两维仍然按矩阵乘法计算，最前面的 `B` 表示 batch。可以理解为有 `B` 对矩阵，每一对独立做一次 `[M, K] @ [K, N]`。

In [ ]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def matmul_batch_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    out.move(pypto.matmul(a, b, pypto.DT_FP32))


def main_matmul_batch():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    a = torch.tensor([[[1, 2], [3, 4]], [[5, 6], [7, 8]]], dtype=torch.float32, device=device)
    b = torch.tensor([[[5, 6], [7, 8]], [[1, 2], [3, 4]]], dtype=torch.float32, device=device)
    out = torch.empty((2, 2, 2), dtype=torch.float32, device=device)

    matmul_batch_kernel(a, b, out)
    ref = torch.matmul(a, b)

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-3, atol=1e-3)
    print("matmul_batch_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("a shape:", tuple(a.shape), "b shape:", tuple(b.shape), "输出 shape:", tuple(out.shape))
    print("输出:", out.cpu())
    print("参考:", ref.cpu())
    print("最大误差:", max_diff)


main_matmul_batch()


`matmul_batch_kernel` 展示 Batch MatMul。输入 shape 都是 `[2, 2, 2]`，输出也是 `[2, 2, 2]`，其中每个 batch 独立计算一个 2x2 矩阵乘法。验证逻辑直接使用 `torch.matmul(a, b)`。

**预期输出说明**

运行成功后，会看到 `matmul_batch_kernel 验证通过`，并打印 batch 输入和输出。输出的第 0 个 batch 对应基础矩阵乘法结果 `[[19, 22], [43, 50]]`。

## 6. Broadcast MatMul：batch 维度可以广播

Broadcast MatMul 的最后两维仍然遵守矩阵乘法规则，前面的 batch 维度按广播规则对齐。例如左输入是 `[1, M, K]`，右输入是 `[B, K, N]`，左输入会在 batch 维度上广播成 `[B, M, K]`。

In [ ]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def matmul_broadcast_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    out.move(pypto.matmul(a, b, pypto.DT_FP32))


def main_matmul_broadcast():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    a = torch.tensor([[[1, 2], [3, 4]]], dtype=torch.float32, device=device)
    b = torch.tensor([[[5, 6], [7, 8]], [[1, 2], [3, 4]]], dtype=torch.float32, device=device)
    out = torch.empty((2, 2, 2), dtype=torch.float32, device=device)

    matmul_broadcast_kernel(a, b, out)
    ref = torch.matmul(a, b)

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-3, atol=1e-3)
    print("matmul_broadcast_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("a shape:", tuple(a.shape), "b shape:", tuple(b.shape), "输出 shape:", tuple(out.shape))
    print("输出:", out.cpu())
    print("参考:", ref.cpu())
    print("最大误差:", max_diff)


main_matmul_broadcast()


`matmul_broadcast_kernel` 展示 batch 维广播。`a.shape=[1, 2, 2]`，`b.shape=[2, 2, 2]`，输出 shape 是 `[2, 2, 2]`。这里不是复制数据的数学定义发生改变，而是 batch 维按广播规则参与计算。

**预期输出说明**

运行成功后，会看到 `matmul_broadcast_kernel 验证通过`，并打印广播后的输出 shape `[2, 2, 2]`。

## 7. 转置输入：a_trans 与 b_trans

有时输入 Tensor 的存放形状不是矩阵乘法想要的逻辑形状。`pypto.matmul` 提供 `a_trans=True` 和 `b_trans=True`，可以在矩阵乘法中按逻辑转置左输入或右输入。

- `b_trans=True`：计算 `a @ b.T`。
- `a_trans=True`：计算 `a.T @ b`。

转置标志不改变 host 侧 Tensor 本身，只改变 matmul 对输入最后两维的解释方式。

In [ ]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def matmul_trans_right_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    out.move(pypto.matmul(a, b, pypto.DT_FP32, b_trans=True))


@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def matmul_trans_left_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    out.move(pypto.matmul(a, b, pypto.DT_FP32, a_trans=True))


def main_matmul_transpose():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    a_right = torch.tensor([[1, 2, 3], [4, 5, 6]], dtype=torch.float32, device=device)
    b_right = torch.tensor([[7, 9, 11], [8, 10, 12]], dtype=torch.float32, device=device)
    out_right = torch.empty((2, 2), dtype=torch.float32, device=device)
    matmul_trans_right_kernel(a_right, b_right, out_right)
    ref_right = torch.matmul(a_right, b_right.transpose(0, 1))

    a_left = torch.tensor([[1, 4], [2, 5], [3, 6]], dtype=torch.float32, device=device)
    b_left = torch.tensor([[7, 8], [9, 10], [11, 12]], dtype=torch.float32, device=device)
    out_left = torch.empty((2, 2), dtype=torch.float32, device=device)
    matmul_trans_left_kernel(a_left, b_left, out_left)
    ref_left = torch.matmul(a_left.transpose(0, 1), b_left)

    right_diff = (out_right - ref_right).abs().max().item()
    left_diff = (out_left - ref_left).abs().max().item()
    torch.testing.assert_close(out_right, ref_right, rtol=1e-3, atol=1e-3)
    torch.testing.assert_close(out_left, ref_left, rtol=1e-3, atol=1e-3)
    print("matmul_transpose_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("右矩阵转置输出:", out_right.cpu(), "参考:", ref_right.cpu())
    print("左矩阵转置输出:", out_left.cpu(), "参考:", ref_left.cpu())
    print("最大误差:", {"right": right_diff, "left": left_diff})


main_matmul_transpose()


这个模块同时验证右矩阵转置和左矩阵转置。右矩阵转置时，`b.shape=[2, 3]` 会被逻辑看成 `[3, 2]`；左矩阵转置时，`a.shape=[3, 2]` 会被逻辑看成 `[2, 3]`。两种写法都得到同一个参考结果 `[[58, 64], [139, 154]]`。

**预期输出说明**

运行成功后，会看到 `matmul_transpose_kernel 验证通过`，并分别打印右矩阵转置输出和左矩阵转置输出。

## 8. 扩展：矩阵乘法加 Bias

真实模型中的线性层通常不是只有矩阵乘法，还会包含 Bias：

```python
out = a @ b + bias
```

如果 `bias` 的形状是 `[N]`，它会广播到输出矩阵 `[M, N]` 的每一行。这是矩阵乘法和逐元素计算组合的一个典型例子。


## 8.1 线性层维度复盘

矩阵乘法加 Bias 可以对应神经网络中的线性层：

```text
y = x @ weight + bias
```

如果从数学广播角度看：

```text
x.shape      = [batch, in_features]
weight.shape = [in_features, out_features]
bias.shape   = [out_features]
```

那么：

```text
y.shape      = [batch, out_features]
```

其中 `in_features` 表示每条输入样本有多少个特征，`out_features` 表示希望生成多少个输出特征。矩阵乘法中间的 `in_features` 维度会被累加掉：

```text
[batch, in_features] @ [in_features, out_features]
                  -> [batch, out_features]
```

在 PyTorch 里，`bias.shape=[out_features]` 通常可以自然广播到每条样本。但 PyPTO 的融合 matmul bias 参数有更具体的工程约束：这里实际传入的 bias 使用二维 `[1, out_features]`，也就是 `[1, N]`。

## 8.2 矩阵乘法加 Bias 的规格

线性层常见表达式是：

```python
out = a @ b + bias
```

| 项目 | 说明 |
| --- | --- |
| `a @ b` | 输出 shape 为 `[M, N]` |
| `bias` | shape 为 `[1, N]`，dtype 为 FP32，广播到输出每一行 |
| `out` | shape 为 `[M, N]`，本节使用 FP32 |
| 计算组合 | Cube 矩阵乘法 + 融合 Bias |

融合 Bias 有两个约束需要显式说明：当输入矩阵是 FP32 时，Bias 必须是 FP32；同时 Bias 需要是二维 Tensor，本节使用 `[1, N]`。这个约束来自底层 matmul 参数检查。

In [ ]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def matmul_bias_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    bias: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    extend_params = {"bias_tensor": bias}
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    out.move(pypto.matmul(a, b, pypto.DT_FP32, extend_params=extend_params))


def main_matmul_bias():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    a = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32, device=device)
    b = torch.tensor([[5, 6], [7, 8]], dtype=torch.float32, device=device)
    bias = torch.tensor([[1, 2]], dtype=torch.float32, device=device)
    out = torch.empty((2, 2), dtype=torch.float32, device=device)

    matmul_bias_kernel(a, b, bias, out)
    ref = torch.matmul(a, b) + bias

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-3, atol=1e-3)
    print("matmul_bias_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("a shape:", tuple(a.shape), "b shape:", tuple(b.shape), "bias shape:", tuple(bias.shape))
    print("输出:", out.cpu())
    print("参考:", ref.cpu())
    print("最大误差:", max_diff)


main_matmul_bias()


`matmul_bias_kernel` 使用 `extend_params={"bias_tensor": bias}` 把 Bias 传给 `pypto.matmul`。数学含义仍然是 `out = a @ b + bias`。

这里需要特别注意融合 Bias 的两个约束：

1. 当矩阵乘法输入是 FP32 时，Bias 必须是 FP32。
2. Bias 必须是二维 Tensor。本节使用 `bias.shape=[1, N]`，它会广播到输出矩阵 `[M, N]` 的每一行。

因此本节中 `a`、`b`、`bias` 和 `out` 都使用 FP32，PyTorch 参考结果也先转成 FP32 后再加 Bias。

这个单元用来验证 `matmul_bias_kernel`。它先在当前设备上构造一组小尺寸输入，再提前分配输出 Tensor 并调用 PyPTO kernel。随后用普通 PyTorch 写出等价的 矩阵乘法 参考结果，打印 shape、样例值和最大误差；最后的 `assert_close` 是正式检查，误差超出阈值时会直接报错。


**预期输出说明**

运行成功后，会看到 `matmul_bias_kernel 验证通过`，并打印矩阵和 Bias 的 shape、dtype。这里应能看到 `bias shape` 是 `(1, N)`，`bias dtype` 和 `out dtype` 都是 `torch.float32`，输出 shape 为 `[M, N]`。

## 9. 课后练习

练习目标是在 `matmul_bias_kernel` 基础上增加 ReLU，完成：

```python
out = maximum(a @ b + bias, 0)
```

思考：

1. 矩阵乘法部分应该使用向量 Tile 还是 Cube Tile？
2. Bias 和 ReLU 属于哪类计算？
3. 为什么 PyPTO 适合把这两类计算写在一个函数中？

参考答案要点：

- 矩阵乘法使用 `set_cube_tile_shapes`。
- Bias 和 ReLU 属于逐元素计算。
- 将完整表达写在一个 JIT 函数中，有利于编译系统看到完整计算链路。


## 10. 可运行扩展：矩阵乘法 + Bias + ReLU

下面给出可直接验证的实现：

```python
out = maximum(a @ b + bias, 0)
```

矩阵乘法部分继续使用 Cube Tile；Bias 作为 matmul 的扩展参数传入，ReLU 属于逐元素计算，需要在 ReLU 前设置 Vec Tile。


In [ ]:
reset_pypto_notebook_state()

@pypto.frontend.jit(runtime_options={"run_mode": RUN_MODE})
def matmul_bias_relu_kernel(
    a: pypto.Tensor([], pypto.DT_FP32),
    b: pypto.Tensor([], pypto.DT_FP32),
    bias: pypto.Tensor([], pypto.DT_FP32),
    out: pypto.Tensor([], pypto.DT_FP32)):
    extend_params = {"bias_tensor": bias}
    pypto.set_cube_tile_shapes([32, 32], [64, 64], [64, 64])
    with_bias = pypto.matmul(a, b, pypto.DT_FP32, extend_params=extend_params)
    pypto.set_vec_tile_shapes(2, 8)
    relu = pypto.maximum(with_bias, 0.0)
    out.move(relu)


def main_matmul_bias_relu():
    if device == "cpu":
        print("当前环境未执行 NPU 验证；NPU 环境中可执行本模块。")
        return

    a = torch.tensor([[1, -2], [3, 4]], dtype=torch.float32, device=device)
    b = torch.tensor([[-5, 6], [7, -8]], dtype=torch.float32, device=device)
    bias = torch.tensor([[1, -2]], dtype=torch.float32, device=device)
    out = torch.empty((2, 2), dtype=torch.float32, device=device)

    matmul_bias_relu_kernel(a, b, bias, out)
    zero = torch.tensor(0.0, dtype=torch.float32, device=device)
    ref = torch.maximum(torch.matmul(a, b) + bias, zero)

    max_diff = (out - ref).abs().max().item()
    torch.testing.assert_close(out, ref, rtol=1e-3, atol=1e-3)
    print("matmul_bias_relu_kernel 验证通过")
    print("device:", device, "run_mode:", RUN_MODE)
    print("输出:", out.cpu())
    print("参考:", ref.cpu())
    print("最大误差:", max_diff)


main_matmul_bias_relu()


这个模块把矩阵乘法、Bias 和 ReLU 写在同一个 JIT kernel 中。Bias 通过 `extend_params={"bias_tensor": bias}` 交给 `pypto.matmul`，得到 `with_bias`；随后 `pypto.set_vec_tile_shapes(2, 8)` 为逐元素 ReLU 设置 Vec Tile，`pypto.maximum(with_bias, 0.0)` 对每个位置做 ReLU，最后通过 `out.move(relu)` 写回输出。


这个验证逻辑使用 PyTorch 写出等价参考结果：`torch.maximum(torch.matmul(a, b).to(torch.float32) + bias, 0)`。两边最大误差在阈值内时，说明融合写法与参考实现一致。


**预期输出说明**

运行成功后，会看到 `matmul_bias_relu_kernel 验证通过`，并打印当前 device、run_mode、输入 shape、Bias shape、输出 shape、最大误差和输出前几项。


核心语句是：

```python
extend_params = {"bias_tensor": bias}
with_bias = pypto.matmul(a, b, pypto.DT_FP32, extend_params=extend_params)
pypto.set_vec_tile_shapes(2, 8)
relu = pypto.maximum(with_bias, 0.0)
out.move(relu)
```


## 11. 本节小结

本节从矩阵乘法形状规则出发，学习了 Cube Tile 的作用，并覆盖了基础 MatMul、Batch MatMul、Broadcast MatMul、输入转置、MatMul + Bias 和 MatMul + Bias + ReLU。后续 Attention Score 中的 `q @ k.T` 本质上也是矩阵乘法，因此本节是学习大模型算子的关键基础。
